# Differentiable coupled model

Using `jax-gcm` as our reference model.

In [ ]:
from importlib.resources import files

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from jcm.model import Model as SpeedyModel
from jcm.forcing import ForcingData
from jcm.terrain import TerrainData
from jcm import load_states_from_xarray
from jcm.physics.speedy.speedy_coords import get_speedy_coords
from jcm.physics.speedy.params import Parameters
from jcm.physics.speedy.speedy_terms import speedy_physics

import warnings
warnings.filterwarnings("ignore")


## 1. JCM setup

In [ ]:
# We define coordinate systems 
coords = get_speedy_coords(spectral_truncation=31, layers=8)

# also load pre-defined terrain and forcing data
data = files("jcm") / "data/bc/t30/clim"
terrain = TerrainData.from_file(data / "terrain.nc", coords=coords)
bc = ForcingData.from_file(data / "forcing.nc", coords=coords, align_mode="wrap_year")

print("Terrain variables: ", terrain.asdict().keys())
print("Boundary variables: ", bc.asdict().keys())

In [ ]:
# Initialize model
model = SpeedyModel(
    coords = coords,
    terrain = terrain, # init terrain (e.g., orography, land mask, etc)
    time_step = 30 # model timestep in minutes
)

In [ ]:
# Run model
result = model.run(
    forcing = bc, 
    total_time = 7, # total model integration time in days
    save_interval = 0.5 # save snapshot in days
)

ds = result.to_xarray()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)

# Plot near-surface T at final timestep
ds.temperature.isel(time=-1, level=0).plot(ax=axes[0], x="lon", y="lat", cmap="viridis", cbar_kwargs={"label": "K"})
axes[0].set(title="Near-surface air temperature", xlabel="Longitude", ylabel="Latitude")

# Plot near-surface q at final timestep
ds.specific_humidity.isel(time=-1, level=0).plot(ax=axes[1], x="lon", y="lat", cmap="viridis", cbar_kwargs={"label": "g/kg"})
axes[1].set(title="Near-surface specific humidity", xlabel="Latitude", ylabel="Latitude");

## 2a. Boundary condition: Run with drier soil

Halve soil moisture availability throughout the year.

In [ ]:
# Halving soil moisture 
dry_bc = bc.copy(
    soilw_am=bc.soilw_am.replace(values=0.5 * bc.soilw_am.values)
)

In [ ]:
# Run
dry = model.run(
    forcing = dry_bc, 
    total_time = 7, 
    save_interval = 0.5
)

dry_ds = dry.to_xarray()

In [ ]:
# Plot evaporation response to drier soil
(dry_ds["surface_flux.evap.0"] - ds["surface_flux.evap.0"]).isel(time=-1).plot(
    x="lon", y="lat", cmap="RdBu_r", center=0,
    cbar_kwargs={"label": "g/m²/s (dry − baseline)"}
)
plt.title("Evaporation response to drier soil");

## 2b. BC sensitivity — forward (JVP)

Question (sensitivity): Estimate the evaporation response to drying after six hours.

In [ ]:
def evaporation(soil_scale):
    forcing = bc.copy(soilw_am=bc.soilw_am.replace(values=soil_scale * bc.soilw_am.values))
    model = SpeedyModel(coords=coords, terrain=terrain, time_step=30)
    result = model.run(forcing=forcing, total_time=0.25, save_interval=0.25) # six-hour run
    ds = result.to_xarray()
    return ds["surface_flux.evap.0"].isel(time=-1).data

In [ ]:
# Start at the original BC; calculate the response to 1% drying.
_, soil_sensitivity = jax.jvp(evaporation, (1.0,), (-0.01,))

In [ ]:
# Plot
soil_map = ds["surface_flux.evap.0"].isel(time=-1).copy(data=np.asarray(soil_sensitivity))
soil_map.plot(
    x="lon", y="lat", cmap="RdBu_r", center=0,
    cbar_kwargs={"label": "Estimated change in evaporation (g/m²/s)"},
)
plt.title("Forward sensitivity: 1% drier soil, after 6 h");

## 3a. Initial condition: Run with warmer temperature

Warm the same starting snapshot by 4 K; compare with an unmodified control.

In [ ]:
# Warming IC
ic = load_states_from_xarray(ds.isel(time=-1).sortby("level"))
warmer_ic = ic.copy(temperature=ic.temperature + 4.0)

In [ ]:
# Matched control from the same snapshot
ic_control = model.run(
    initial_state = ic, 
    forcing = bc, 
    total_time = 7, 
    save_interval = 0.5
)
control_ds = ic_control.to_xarray()

# Run perturbed model
ic_result = model.run(
    initial_state = warmer_ic, 
    forcing = bc, 
    total_time = 7, 
    save_interval = 0.5
)

ic_ds = ic_result.to_xarray()

In [ ]:
# Plot evaporation response to warmer atmosphere
(ic_ds["surface_flux.evap.0"] - control_ds["surface_flux.evap.0"]).isel(time=-1).plot(
    x="lon", y="lat", cmap="RdBu_r", center=0,
    cbar_kwargs={"label": "g/m²/s (warmer − control)"}
)
plt.title("Evaporation response to warmer temperature");

## 3b. Parameter sensitivity — reverse (VJP)

Question (derivative): compute **dJ/d(theta)**, where J is the mean land evaporation and thetas are parameters. 

In [ ]:
parameters = Parameters.default()
defaults = parameters.surface_flux
land = np.asarray(terrain.fmask) > 0

In [ ]:
# Define J (mean land evaporation)
def J(chl, cdl, vgust, fstab, clambda):
    surface = defaults.replace(chl=chl, cdl=cdl, vgust=vgust, fstab=fstab, clambda=clambda)
    physics = speedy_physics(parameters.replace(surface_flux=surface))
    model = SpeedyModel(coords=coords, terrain=terrain, physics=physics, time_step=30)
    result = model.run(initial_state=ic, forcing=bc, total_time=0.25, save_interval=0.25)
    ds = result.to_xarray()
    return ds["surface_flux.evap.0"].isel(time=-1).data[land].mean()

In [ ]:
# Reverse sensitivity at the default parameter values
J0, reverse = jax.vjp(
    J, defaults.chl, defaults.cdl, defaults.vgust, defaults.fstab, defaults.clambda
)
parameter_sensitivity = jnp.array(reverse(1.0)) # starts from dJ/dJ = 1.0 --> dJ/d(theta)
names = ["chl", "cdl", "vgust", "fstab", "clambda"]
units = ["dimensionless unit", "dimensionless unit", "(m/s)", "dimensionless unit", "(W/m²/K)"]
for name, value, unit in zip(names, parameter_sensitivity, units):
    print(f"dJ/d{name:<7} = {value: .3g} g/m²/s per {unit}")